# Biopython으로 라이노바이러스 서열 분석해보기
## 개요
아니 얘는 또 뭐 하는 앱니까? 여러분들, 감기랑 독감이랑 원인균 다른거 아셨습니까? 독감은 인플루엔자가 범인이고 일반 감기는 라이노바이러스가 원인입니다. 

## 감기에는 진짜로 약이 없다
아니 뭔 소리임? 우리 감기약 먹잖아요! 그건 증상을 완화시켜서 면역계가 바이러스를 쫓아내는 걸 **도와주는**겁니다. 감기 바이러스를 직접 공격하는 약은 없어요. 왜냐고요? 감기 바이러스는 변이율이 진짜 장난아니거든요! 눈 깜빡할 새에 변이합니다. 그래서 감기약은 바이러스를 공격하는 원딜이 아니라 우리편을 도와주는 서폿입니다. 

## 프로젝트 정보
- 인원: 1인(개인 프로젝트)
- 버전: 3.10(TF_base)
- 설치할 것들: Biopython, muscle
- 데이터 리소스: NCBI(Entrez로 갖고올 예정)

In [ ]:
# 모듈
import numpy as np
import matplotlib.pyplot as plt
import math

# Biopython
from Bio import Entrez, SeqIO # 왼쪽: 일단 털어보자/오른쪽: 시퀀스 다루려면 필요합니다. 필수임. 
from Bio import AlignIO # 서열 분석해줄 친구
from Bio import Phylo # 트리 그릴라면 필요해요 
from Bio.Align import AlignInfo
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
from Bio.Seq import Seq

import io # 누구세요?
import subprocess # 서브 프로세스(이건 또 뭐여...)
from collections import defaultdict
from collections import Counter

# 통계분석용
from scipy.stats import mannwhitneyu
from itertools import combinations

In [ ]:
# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Nanumsquare_ac' # 나눔바른펜(본인 기본 고딕 싫어함)
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 사전세팅
Entrez.email = "blackholekun@gmail.com" # 이메일 
muscle_exe = "/opt/homebrew/bin/muscle" # 이거 경로 있어야 써요(which 치면 나옴)

# 창고가 열려있으면 털어주는게 인지상정! 
- 여기서는 라이노바이러스의 VP1로 진행하겠습니다. 

In [ ]:
# 바이러스 서열 다운로드
virus_query = "Rhinovirus[Organism] AND VP1 AND complete cds AND 7000:7500[Sequence Length]"

print('Searching sequences... ') # 솔직히 이거 없으면 되는건지 불안하잖아요...

handle = Entrez.esearch(db="nucleotide", term=virus_query, retmax=50)
record = Entrez.read(handle)
id_list = record['IdList']

print(f"총 {len(id_list)}개의 표준 서열을 찾았습니다.")

In [ ]:
# 콤퓨타에 저-장
vir_sequence = []
for i, id in enumerate(id_list):
    print(f"Downloading sequence {i+1}/{len(id_list)}: {id}")
    handle = Entrez.efetch(db="nucleotide", id=id, rettype="fasta", retmode="text")
    record = SeqIO.read(handle, "fasta")
    
    # record에 id와 seq가 다 들어가야되더라... (안되면 오류남 봤음)
    vir_sequence.append(record) 

# 파일로 저장
SeqIO.write(vir_sequence, "rhinovirus_sequence_20.fasta", "fasta")
print('Done!')

In [ ]:
# 시퀀스 길이 체크 
for rec in vir_sequence:
    print(f"ID: {rec.id} | Length: {len(rec.seq)}")

# MSA
- 머슬 썼습니다. (clustalW 아마 깔려는 있을듯)

In [ ]:
print('MSA start... ')

# MSA 분석 시-작
try: 
    result = subprocess.run([muscle_exe, "-align", "rhinovirus_sequence_20.fasta", "-output", "rhinovirus_muscle_aligned.fasta"], check=True, capture_output=True, text=True)
    print("Completed. ")
except subprocess.CalledProcessError as e: 
    print(f"MSA failed: {e}")
finally:
    alignment = AlignIO.read("rhinovirus_muscle_aligned.fasta", "fasta")

# 밥 먹고 오면 끝나있겠는데...? 

In [ ]:
print("====== MSA Result ======")
alignment = AlignIO.read("rhinovirus_muscle_aligned.fasta", "fasta") # FASTA 니네 확장자가 몇개냐... 

for record in alignment:
    print(f"{record.id[:10]:<15} : {record.seq[:100]}")

In [ ]:
def calculate_conservation_no_gap(alignment, gap_threshold=0.5):
    length = alignment.get_alignment_length()
    scores = []

    for i in range(length):
        column_raw = alignment[:, i]

        # gap 비율이 너무 높으면 제외 (선택사항)
        gap_fraction = column_raw.count("-") / len(column_raw)
        if gap_fraction > gap_threshold:
            continue

        # gap 제거
        column = column_raw.replace("-", "")
        if len(column) == 0:
            continue

        # 최빈 염기 비율 = 보존도
        most_common = max(set(column), key=column.count)
        score = column.count(most_common) / len(column)
        scores.append(score)

    return scores

scores = calculate_conservation_no_gap(alignment)

print(f"해당 구간의 평균 보존율: {np.mean(scores)*100:.2f}%")

# 섀넌 앤트로피

In [ ]:
def calculate_shannon_entropy(alignment, gap_threshold=0.5):
    length = alignment.get_alignment_length()
    entropies = []

    for i in range(length):
        column_raw = alignment[:, i]
        
        # gap 비율 계산
        gap_fraction = column_raw.count("-") / len(column_raw)
        if gap_fraction > gap_threshold:
            continue  # gap 많은 position 제거
        
        # gap 제거
        column = column_raw.replace("-", "")
        if len(column) == 0:
            continue
        
        counts = Counter(column)
        total = sum(counts.values())
        
        entropy = 0
        for c in counts.values():
            p = c / total
            entropy -= p * np.log2(p)
        
        entropies.append(entropy)

    return entropies

def sliding_window_mean(values, window=20):
    """
    values : np.array (entropy scores, np.nan 포함)
    window : window size
    """
    smoothed = []

    for i in range(len(values)):
        start = max(0, i - window // 2)
        end = min(len(values), i + window // 2 + 1)

        window_vals = values[start:end]
        window_vals = window_vals[~np.isnan(window_vals)]

        if len(window_vals) == 0:
            smoothed.append(np.nan)
        else:
            smoothed.append(np.mean(window_vals))

    return np.array(smoothed)

In [ ]:
entropies = calculate_shannon_entropy(alignment)

H_max = np.log2(4)
variation_scores = [e / H_max for e in entropies]  # 0~1 스케일

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(variation_scores, alpha=0.8, color='black')
plt.fill_between(range(len(variation_scores)), variation_scores, alpha=0.3)

plt.axvline(751, linestyle='--', color='red', alpha=0.5)
plt.axvline(745, linestyle='--', color='red', alpha=0.5)
plt.text(751, max(variation_scores)*1.07, 'Spike POS 751, 745', color='red', alpha=0.5, ha='center', fontweight='bold')

plt.axvline(2214, linestyle='--', color='red', alpha=0.5)
plt.axvline(2202, linestyle='--', color='red', alpha=0.5)
plt.text(2214, max(variation_scores)*1.07, 'Spike POS 2202, 2214', color='red', alpha=0.5, ha='center', fontweight='bold')

plt.title("Viral Variation Hotspots", y = 1.07)
plt.xlabel("Alignment Position (filtered)")
plt.ylabel("Normalized Shannon Entropy")
plt.show()

- 한타바이러스거 그대로 갖다썼더니 풀로 안보여주데... 

## 변이 핫스팟

In [ ]:
# 섀넌 엔트로피 점수 도출
def get_top_variable_sites_no_gap(alignment, top_n=10):
    length = alignment.get_alignment_length()
    variability = []

    ref_seq = alignment[0].seq

    for i in range(length):
        # 🔴 reference가 gap이면 무조건 스킵
        if ref_seq[i] == '-':
            continue

        column = alignment[:, i].replace("-", "")
        if not column:
            continue

        counts = Counter(column)
        total = sum(counts.values())

        entropy = 0.0
        for c in counts.values():
            p = c / total
            entropy -= p * math.log2(p)

        variability.append((i, entropy))

    return sorted(variability, key=lambda x: x[1], reverse=True)[:top_n]

def alignment_to_sequence_pos(aligned_seq, aln_pos):
    count = 0
    for i in range(aln_pos + 1):
        if aligned_seq[i] != '-':
            count += 1
    return count


ref_seq = alignment[0].seq
top_sites = get_top_variable_sites_no_gap(alignment, top_n=10)

high_entropy_ha_sites = []

print("--- 변이가 집중된 주요 포지션 분석 결과 ---")
for aln_pos, score in top_sites:
    real_pos = alignment_to_sequence_pos(ref_seq, aln_pos)
    high_entropy_ha_sites.append(real_pos)
    print(f"Alignment {aln_pos:4d} → Pos {real_pos:4d} | 엔트로피: {score:.3f}")

print("\n최종 고엔트로피 포지션 리스트:")
print(high_entropy_ha_sites)

## 통계분석
- 귀무가설: 라이노바이러스의 변이는 무작위적으로 발생하며, 특정 위치에 선호적으로 집중되지 않는다.
- 대립가설: 라이노바이러스의 변이는 무작위가 아니며, 특정 위치(hotspots)에 유의하게 집중된다.

In [ ]:
entropy_raw = np.array(entropies)

mean_raw = np.mean(entropy_raw)
median_raw = np.median(entropy_raw)
iqr_raw = np.percentile(entropy_raw, 75) - np.percentile(entropy_raw, 25)

print("[Raw entropy]")
print(f"Mean:   {mean_raw:.4f}")
print(f"Median: {median_raw:.4f}")
print(f"IQR:    {iqr_raw:.4f}")

In [ ]:
plt.hist(entropy_raw, bins=50, color='black')
plt.title("Raw Shannon Entropy Distribution (Site-wise)")
plt.xlabel("Entropy (bits)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Windowed entropy
entropy_window = sliding_window_mean(entropy_raw, window=25)
entropy_window_valid = entropy_window[~np.isnan(entropy_window)]

mean_win = np.mean(entropy_window_valid)
median_win = np.median(entropy_window_valid)
iqr_win = (
    np.percentile(entropy_window_valid, 75)
    - np.percentile(entropy_window_valid, 25)
)

print("[Windowed entropy]")
print(f"Mean:   {mean_win:.4f}")
print(f"Median: {median_win:.4f}")
print(f"IQR:    {iqr_win:.4f}")

In [ ]:
plt.hist(entropy_window_valid, bins=50, color='black')
plt.title("Windowed Shannon Entropy Distribution (Regional)")
plt.xlabel("Mean entropy (windowed)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# --- 2. normalization ---
entropy_min = np.nanmin(entropy_window)
entropy_max = np.nanmax(entropy_window)

entropy_window_normalized = (
    entropy_window - entropy_min
) / (entropy_max - entropy_min)

variation_scores = entropy_window_normalized

# --- 3. NaN 제거 (🔥 중요) ---
valid_scores = variation_scores[~np.isnan(variation_scores)]

# --- 4. hotspot threshold (normalized 기준) ---
threshold_norm = np.percentile(valid_scores, 90)
threshold_raw = threshold_norm * (entropy_max - entropy_min) + entropy_min

hotspots = valid_scores[valid_scores >= threshold_norm]
non_hotspots = valid_scores[valid_scores < threshold_norm]

u_stat, p_value = mannwhitneyu(
    hotspots,
    non_hotspots,
    alternative="greater"
)

print(f"Hotspot threshold (top 10%, normalized entropy): {threshold_norm:.3f}")
print(f"Corresponding raw entropy threshold: {threshold_raw:.3f}")
print(f"Mann–Whitney U statistic: {u_stat:.1f}")
print(f"p-value: {p_value:.4e}" if p_value > 1e-10 else "p-value: <1e-10")

### 시각화

In [ ]:
# 변이 점수 분포 히스토그램 (NanumSquare 적용)
plt.figure(figsize=(10, 6))
plt.hist(entropy_raw, bins=50, color='#34495e', edgecolor='white', alpha=0.8)

# 통계 지표 수직선 표시
plt.axvline(mean_raw, color='red', linestyle='dashed', linewidth=1, label=f'Mean: {mean_raw:.4f}')
plt.axvline(median_raw, color='orange', linestyle='dashed', linewidth=1, label=f'Median: {median_raw:.4f}')

plt.title('라이노바이러스 변이 점수 분포 (Entropy Distribution)', fontsize=15)
plt.xlabel("Variation Score (Entropy)", fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()


# Phylogenic tree

In [ ]:
# 1. 거리 행렬 계산 (Identity 모델 사용)
calculator = DistanceCalculator('identity')
dm = calculator.get_distance(alignment)

# 2. Neighbor-Joining(NJ) 트리 생성
constructor = DistanceTreeConstructor(calculator, 'nj')
tree = constructor.build_tree(alignment)
tree.root_at_midpoint() # 루트를 중간으로 잡아 균형 잡힌 트리 생성

# 3. 시각화 (NanumSquare 폰트가 이미 글로벌 설정되어 있으므로 바로 출력!)
fig = plt.figure(figsize=(12, 15), dpi=100)
ax = fig.add_subplot(1, 1, 1)
plt.title("Rhinovirus Phylogenetic Tree (Based on Whole Genome)", fontsize=18, pad=20)

# Bio.Phylo를 이용한 트리 드로잉
Phylo.draw(tree, axes=ax, do_show=False, label_func=lambda n: str(n) if n.is_terminal() else "")

# 후처리: 축 숨기기 등 깔끔하게 정리
plt.axis('off')
plt.tight_layout()
plt.show()

## 통계분석
- 귀무가설: 동일 clade 내 서열 유사도와 서로 다른 clade 간 서열 유사도에는 차이가 없다.
- 대립가설: 동일 clade 내 서열 유사도가 clade 간 서열 유사도보다 유의하게 높다.

In [ ]:
def pairwise_identity(seq1, seq2):
    matches = sum(a == b for a, b in zip(seq1, seq2) if a != '-' and b != '-')
    length = sum(a != '-' and b != '-' for a, b in zip(seq1, seq2))
    return matches / length if length > 0 else 0

def extract_clades(tree, cutoff=0.05):
    clade_map = {}
    clade_id = 0

    for clade in tree.find_clades():
        if clade.branch_length and clade.branch_length > cutoff:
            terminals = clade.get_terminals()
            for t in terminals:
                clade_map[t.name] = f"Clade_{clade_id}"
            clade_id += 1

    return clade_map
clade_map = extract_clades(tree, cutoff=0.05)

# ID 정규화 (이거 중요)
normalized_clade_map = {}
for k, v in clade_map.items():
    normalized_clade_map[k.split('.')[0]] = v

In [ ]:
within_clade = []
between_clade = []

for rec1, rec2 in combinations(alignment, 2):
    id1 = rec1.id.split('.')[0]
    id2 = rec2.id.split('.')[0]

    if id1 not in normalized_clade_map or id2 not in normalized_clade_map:
        continue

    identity = pairwise_identity(str(rec1.seq), str(rec2.seq))

    if normalized_clade_map[id1] == normalized_clade_map[id2]:
        within_clade.append(identity)
    else:
        between_clade.append(identity)

u, p = mannwhitneyu(
    within_clade,
    between_clade,
    alternative="greater"
)

print(f"Within-clade pairs: {len(within_clade)}")
print(f"Between-clade pairs: {len(between_clade)}")
print(f"U-statistic: {u}")
print(f"p-value: {p:.4e}" if p > 1e-10 else "p-value: <1e-10")

### 이펙트 사이즈

In [ ]:
effect_size = (
    np.median(within_clade) - np.median(between_clade)
)

print(f'effect_size: {effect_size:.4f}')

In [ ]:
n1 = len(within_clade)
n2 = len(between_clade)

rbc = 1 - (2 * u) / (n1 * n2)
print(f"Rank-biserial r: {rbc:.4f}")